# Setup
## Libraries and functions

In [ ]:
pkgs <- c('data.table','ggplot2','ggrepel','gtExtras','lme4','lme4qtl','lmerTest','lmtest','mediation','parallel','patchwork','geomtextpath')
if(!require(lme4qtl)) { remotes::install_github("variani/lme4qtl") } # Special install
lapply(pkgs, \(pkg) { if(!require(pkg, character.only=T)) {install.packages(pkg);require(pkg)}}) |> invisible()

options(datatable.na.strings=c('NA',''))

# Workaround so that lmerTest accepts lme4qtl model objects, so we can get lmerTest Satterthwaite p-values.
myLmer <- \(formula, data, relmat) {
  model <- relmatLmer(formula, data, relmat=relmat)
  lmerTest:::as_lmerModLT(model, as.function(model))
}

In [ ]:
ws_bucket <- Sys.getenv('WORKSPACE_BUCKET')
system('mkdir -p data/derived/genomics')
system('mkdir -p data/derived/metabolomics/QCd/')
grab_file_if_not_extant <- \(f) if(!file.exists(f)) system(paste0('gcloud storage cp', ws_bucket,'/',f))
grab_file_if_not_extant('data/derived/analysis_df-fhs.csv')
grab_file_if_not_extant('data/derived/genomics/fhs_km.RData')
grab_file_if_not_extant('data/derived/metabolomics/QCd/merged_QCd-aligned.csv')
grab_file_if_not_extant('data/derived/metabolomics/met_info-aligned.csv')

In [ ]:
data <- fread('data/derived/analysis_df-fhs.csv')
variants_of_interest <- fread(cmd='gcloud storage cat gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/variants_of_interest.csv')
cpaids <- variants_of_interest[cpaid %in% names(data), cpaid]

data <- fread('data/derived/analysis_df-fhs.csv')[
  ][, metabolomics_visit := as.factor(metabolomics_visit)
  # Variants are numeric (0/1/2), but we temporarily set to character so they don't get scale()'d.
  ][, names(.SD) := lapply(.SD, as.character), .SDcols=cpaids
  ][, names(.SD) := lapply(.SD, scale       ), .SDcols=is.numeric
  ][, names(.SD) := lapply(.SD, as.numeric  ), .SDcols=cpaids
]

load('data/derived/genomics/fhs_km.RData') # Kinship (with ids remapped in script 01c)
data <- data[NWD_Id %in% colnames(km)]

met_nms <- names(fread('data/derived/metabolomics/QCd/merged_QCd-aligned.csv',nrows=0,drop=1))
# Note: FHS does not have batch_cp/_cn/_hp/_an covars like MESA, only `metabolomics_visit`.
covars <- c('sex', 'age', 'smoking', 'metabolomics_visit', paste0('gPC',1:9))
outcomes <- c('hdl_log', 'tg_log', 'fg')
exposures <- c('bmi','sex')

calc_eff_n_metabolites <- \(met_nms) {
  met_nms <- met_nms[!is.na(met_nms)] |> unique()
  met_mtx <- data[ rowSums(is.na(data[,..met_nms]))==0, ..met_nms ] |> as.matrix(rownames=1) # Only samples having data for ALL metabolomics methods
  met_eigvals <- prcomp(met_mtx, scale=T, center=T)$sdev^2
  met_eff_n <- sum(met_eigvals)^2 / sum(met_eigvals^2)
}

# Map aligned vs. MESA-specific metabolites
We only need to rerun the metabolites that were already significant in MESA.

First, a sanity check that the QI ids to indeed correspond between the MESA & align metabolomics datasets.

In [ ]:
met_info_mesa  <- fread('data/derived/metabolomics/met_info-mesa.csv'   )
met_info_align <- fread('data/derived/metabolomics/met_info-aligned.csv')

# Sanity check that the QI ids do correspond between the MESA & aligned metabolomics datasets
tmp1 <- met_info_align[Method!='Amide-neg'][met_info_mesa[Method!='Amide-neg'], on=.(unique_met_id)]
tmp2 <- met_info_align[Method=='Amide-neg'][met_info_mesa[Method=='Amide-neg'], on=.(  Metabolite )][, unique_met_id := i.unique_met_id] # I manually generated the unique ids for the amines arbitrarily. Must match on name instead.
tmp <- rbind(tmp1,tmp2,fill=T)
tmp[, all(      Compound_ID==      i.Compound_ID | is.na(      Compound_ID))] # [1] TRUE
tmp[, all(Pilot_Compound_ID==i.Pilot_Compound_ID | is.na(Pilot_Compound_ID))] # [1] TRUE

Not all metabolites in the MESA dataset are present in the aligned dataset. We use the best-correlated metabolite as proxy.

In [ ]:
mesa_met_data <- fread('data/derived/metabolomics/QCd/merged_QCd-mesa.csv')[,-'TOM_Id']
met_corrs <- cor(mesa_met_data, use='complete.obs')
met_corrs <- met_corrs_cpy
met_corrs <- met_corrs[,colnames(met_corrs) %in% met_info_align$unique_met_id]

mesa_signif_GxMs <- fread('mesa_signif_GxMs.csv')
met_mapping <- data.table(mesa_met_id = mesa_signif_GxMs$M)
met_mapping[, aligned_met_id := sapply(mesa_met_id, \(m) colnames(met_corrs)[  which.max(met_corrs[m,])])]
met_mapping[, corr           := sapply(mesa_met_id, \(m)          met_corrs [m,which.max(met_corrs[m,])])]
met_mapping

# Rerun GxMs which were signif in MESA
## Construct formulas
Model: `Y ~ G*M + G*E + covariates + gPCs*M + (1|ID)` \
Except, we do not have an mvpa variable in FHS, so in this case GxE is excluded.

In [ ]:
GxMpGxE_models <-
CJ(Y=mesa_signif_GxMs$Y, E=mesa_signif_GxMs$E, G=mesa_signif_GxMs$G, M=mesa_signif_GxMs$M)[
  ][,unique(.SD)
  ][mapply(Y,E,G,M, FUN=\(Y,E,G,M) # Only keep signif GxMs from the regular analysis.
      any(Y==mesa_signif_GxMs$Y & E==mesa_signif_GxMs$E & G==mesa_signif_GxMs$G & M==mesa_signif_GxMs$M))
  ][met_mapping, on=.(M=mesa_met_id)
  ][, M := aligned_met_id
  ][, fmla := paste0('<Y> ~ <G>*<M>+<G>*<E>')            # Construct formulas.
  ][E=='mvpa_wins', fmla := paste0('<Y> ~ <G>*<M>')      #   Except we don't have an MVPA variable for FHS, so exclude <E> in that case.
  ][, fmla := paste(fmla,'+',paste(covars,collapse='+')) # Add covars.
  ][, fmla := paste(fmla,'+',                            # Add G*covars.
        paste(paste0('<G>*',covars),collapse=' + '))     #
  ][, fmla := gsub('\\+ <G>\\*gPC[0-9]','',fmla)         #   Rm <G>*gPC terms though.
  ][, fmla := paste(fmla,'+',                            # Add gPC*M covars.
        paste(collapse='+', paste0('gPC',1:9,'*<M>')))
  ][, fmla := mapply(Y,E,G,M,fmla, FUN=\(Y,E,G,M,fmla) { # Replace placeholders.
        fmla <- gsub('<Y>',Y,fmla)
        fmla <- gsub('<E>',E,fmla)
        fmla <- gsub('<G>',G,fmla)
        fmla <- gsub('<M>',M,fmla) })
  ][, fmla := paste(fmla,'+ (1|NWD_Id)')                # Add random intercept.
  ][, term_pat := paste0(G,':',M,'|',M,':',G)            # Term whose β to extract.
]

## Run models

In [ ]:
options(mc.cores=6L)
i <- 0
t0 <- Sys.time()
GxMpGxE_models[
  ][, c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar.
        model <- myLmer(fmla,data,list(NWD_Id=km))                           # Run LMM model.
        coefs <- summary(model)$coefficients                                 # Extract coefs.
        c( coefs[grepl(pat,rownames(coefs)),], nrow(model@frame) )
      }))
]
Sys.time() - t0

## Display results

In [ ]:
fread('data/derived/metabolomics/met_info-aligned.csv')[
  ][GxMpGxE_models, on=.(unique_met_id=M) # Merge in met metadata.
  ][variants_of_interest, on=.(G=cpaid)   # Merge in SNP metadata.
  ][!is.na(est)                           # Discard SNPs which weren't run.
  ][, .(Y,G,rsid,analysis,β_GxM=est,se,p, # Keep only desired columns.
        M=unique_met_id, Metabolite,proxy_pearson_corr=corr)
  ][, names(.SD) := lapply(.SD,signif,3)      # Keep only 3 signif digits.
    , .SDcols=is.numeric,               
  ][order(p)
] |>
  dplyr::group_by(G,rsid,analysis) |>
  gt(caption='GxMs') |>
  gt:::as.tags.gt_tbl()